[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG952/blob/main/materials/week09/notebooks/AG952_W9_BrewDog_Facilitator.ipynb)

# AG952 | Workshop 9 Facilitator Debrief — BrewDog Narrative & Transformer Analysis

*Dr James Bowden, Strathclyde Business School*

---

Run after the session ends. Pulls from Google Sheets. Two parts: **Decision Audit** + **Results Analysis**.

**Prerequisites:** Google Sheet populated with student data; service account credentials JSON stored in Colab Secrets as `GSHEET_CREDS`.

In [ ]:
#@title Configuration — set these before pulling data
SPREADSHEET_ID  = ""       # paste Google Sheet ID here
WORKSHEET_NAME  = "Sheet1"
APPS_SCRIPT_URL = ""      # same URL as student notebook

In [ ]:
#@title Step 0 — Install and import
!pip install gspread google-auth pandas numpy matplotlib seaborn wordcloud -q
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import json, re
from wordcloud import WordCloud
from google.colab import userdata
from google.oauth2.service_account import Credentials
import gspread
from IPython.display import display, Markdown, HTML
import warnings; warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})
print("Ready.")

In [ ]:
#@title Pull data from Google Sheets
creds_json = userdata.get("GSHEET_CREDS")
creds_dict = json.loads(creds_json)
scopes = ["https://www.googleapis.com/auth/spreadsheets.readonly",
          "https://www.googleapis.com/auth/drive.readonly"]
creds = Credentials.from_service_account_info(creds_dict, scopes=scopes)
gc = gspread.authorize(creds)
ws = gc.open_by_key(SPREADSHEET_ID).worksheet(WORKSHEET_NAME)
rows = ws.get_all_records()
df = pd.DataFrame(rows)
# Clean numeric columns
for col in ["cp6_finbert_accuracy","cp6_distilbert_accuracy","cp4_sentiment_2010_2014","cp4_sentiment_2019_2025"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
print(f"Loaded {len(df)} team submissions.")
display(df.head())

## Part 1 — Decision Audit

In [ ]:
#@title 1.1 — Full decision overview
# Show a styled table of all team decisions
cols = ["team_name","cp1_period","cp2_normalisation","cp2_stopwords",
        "cp3_n_topics","cp4_dictionary","cp6_model","cp7_interpretability_choice"]
avail = [c for c in cols if c in df.columns]
display(df[avail].to_html(index=False))
# Print summary counts
for col in ["cp2_normalisation","cp2_stopwords","cp4_dictionary","cp6_model"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().to_string())

In [ ]:
#@title 1.2 — Pre-processing decision heatmap
if "cp2_normalisation" in df.columns and "cp2_stopwords" in df.columns:
    ct = pd.crosstab(df["cp2_normalisation"], df["cp2_stopwords"])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(ct, annot=True, fmt="d", cmap="YlOrRd", ax=axes[0], cbar=False)
    axes[0].set_title("Pre-processing choices: normalisation \u00d7 stop-words\n(cell = number of teams)")
    axes[0].set_xlabel("Stop-word list"); axes[0].set_ylabel("Normalisation")
    
    # Stacked bar: dictionary choice by normalisation
    if "cp4_dictionary" in df.columns:
        ct2 = pd.crosstab(df["cp2_normalisation"], df["cp4_dictionary"])
        ct2.plot(kind="bar", stacked=True, ax=axes[1], colormap="tab10", edgecolor="white")
        axes[1].set_title("Dictionary choice by normalisation method")
        axes[1].set_xlabel("Normalisation"); axes[1].set_ylabel("Teams")
        axes[1].tick_params(axis="x", rotation=25)
        axes[1].legend(title="Dictionary", fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
#@title 1.3 — LDA topic count choices (k)
if "cp3_n_topics" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df["cp3_n_topics_num"] = pd.to_numeric(df["cp3_n_topics"], errors="coerce")
    axes[0].hist(df["cp3_n_topics_num"].dropna(), bins=range(2,14), color="#4f46e5",
                 edgecolor="white", alpha=0.85)
    axes[0].set_title("Distribution of k (number of LDA topics) across teams")
    axes[0].set_xlabel("k"); axes[0].set_ylabel("Teams")
    
    # Model choice pie
    if "cp6_model" in df.columns:
        model_counts = df["cp6_model"].value_counts()
        axes[1].pie(model_counts.values, labels=[m.split(" (")[0] for m in model_counts.index],
                    autopct="%1.0f%%", colors=["#6366f1","#10b981","#f59e0b"])
        axes[1].set_title("Transformer model selection")
    plt.tight_layout(); plt.show()
    print(f"\nMedian k: {df['cp3_n_topics_num'].median():.0f}")
    print(f"Range: {df['cp3_n_topics_num'].min():.0f} \u2013 {df['cp3_n_topics_num'].max():.0f}")

In [ ]:
#@title 1.4 — Student interpretability stance distribution
if "cp7_interpretability_choice" in df.columns:
    # Simplify labels
    def simplify(s):
        if "Dictionary" in str(s): return "Dictionary (transparent)"
        if "Transformer" in str(s): return "Transformer (trust model)"
        return "Neither (use both + human review)"
    df["interp_simplified"] = df["cp7_interpretability_choice"].apply(simplify)
    counts = df["interp_simplified"].value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(counts.index, counts.values, color=["#22c55e","#ef4444","#f59e0b"][:len(counts)],
                   edgecolor="white")
    ax.set_title("Student interpretability stance (CP7)")
    ax.set_xlabel("Teams"); ax.set_xlim(0, counts.max() + 1)
    for bar, v in zip(bars, counts.values):
        ax.text(v + 0.1, bar.get_y() + bar.get_height()/2, str(v), va="center", fontsize=11)
    plt.tight_layout(); plt.show()
    print("\nDiscussion prompt: Does the choice of model in CP6 predict the interpretability")
    print("stance in CP7? Check whether FinBERT teams were more or less likely to choose 'Trust model'.")

## Part 2 — Results Analysis

In [ ]:
#@title 2.1 — Reported sentiment scores: early vs crisis period
cols_sent = ["cp4_sentiment_2010_2014","cp4_sentiment_2019_2025"]
avail_sent = [c for c in cols_sent if c in df.columns and df[c].notna().any()]
if avail_sent:
    sent_df = df[["team_name"] + avail_sent + (["cp4_dictionary"] if "cp4_dictionary" in df.columns else [])].dropna(subset=avail_sent)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scatter: early vs crisis, coloured by dictionary
    if "cp4_dictionary" in df.columns:
        dicts = sent_df["cp4_dictionary"].unique()
        colors = dict(zip(dicts, ["#4f46e5","#f59e0b","#10b981","#ef4444"]))
        for d, grp in sent_df.groupby("cp4_dictionary"):
            axes[0].scatter(grp["cp4_sentiment_2010_2014"], grp["cp4_sentiment_2019_2025"],
                           label=d.split(" (")[0], color=colors.get(d,"grey"), s=80, alpha=0.8)
    else:
        axes[0].scatter(sent_df["cp4_sentiment_2010_2014"], sent_df["cp4_sentiment_2019_2025"],
                       color="#4f46e5", s=80)
    axes[0].axhline(0, color="grey", lw=0.8, linestyle="--")
    axes[0].axvline(0, color="grey", lw=0.8, linestyle="--")
    axes[0].set_xlabel("Mean VADER score 2010\u20132014"); axes[0].set_ylabel("Mean VADER score 2019\u20132025")
    axes[0].set_title("Sentiment scores: growth era vs crisis\n(each point = one team)")
    if "cp4_dictionary" in df.columns: axes[0].legend(title="Dictionary", fontsize=8)
    
    # Bar: mean scores by period
    means = sent_df[avail_sent].mean()
    axes[1].bar(["2010\u20132014 (growth)", "2019\u20132025 (crisis)"], means.values,
                color=["#22c55e","#ef4444"], edgecolor="white", width=0.5)
    axes[1].axhline(0, color="grey", lw=0.8, linestyle="--")
    axes[1].set_title("Class mean sentiment by period")
    axes[1].set_ylabel("Mean VADER compound score")
    for i, v in enumerate(means.values):
        axes[1].text(i, v + 0.005, f"{v:+.3f}", ha="center", fontsize=11)
    plt.tight_layout(); plt.show()

In [ ]:
#@title 2.2 — Transformer model accuracy across teams
acc_cols = {c: c.replace("cp6_","").replace("_accuracy","").replace("_"," ").title()
            for c in ["cp6_finbert_accuracy","cp6_distilbert_accuracy"] if c in df.columns}
if acc_cols:
    fig, ax = plt.subplots(figsize=(10, 4))
    for col, label in acc_cols.items():
        vals = df[col].dropna()
        if len(vals): ax.hist(vals, bins=10, alpha=0.7, label=label, edgecolor="white")
    ax.set_title("Distribution of transformer accuracy scores across teams")
    ax.set_xlabel("Accuracy on ground-truth labels"); ax.set_ylabel("Teams")
    ax.legend()
    plt.tight_layout(); plt.show()
    print("\nMean accuracy by model:")
    for col, label in acc_cols.items():
        print(f"  {label}: {df[col].mean():.2%} (n={df[col].notna().sum()} teams)")

In [ ]:
#@title 2.3 — Analyst note word cloud
if "cp8_analyst_note" in df.columns:
    notes = " ".join(df["cp8_analyst_note"].dropna().tolist())
    if notes.strip():
        wc = WordCloud(width=900, height=350, background_color="white",
                       colormap="RdYlGn", max_words=80,
                       stopwords={"the","a","an","and","of","to","in","is","are","was","were",
                                  "for","that","this","with","have","has","had","be","been","but",
                                  "we","our","their","would","could","should","may","might","also"}).generate(notes)
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
        ax.set_title("Analyst notes \u2014 word cloud (all teams)", fontsize=13)
        plt.tight_layout(); plt.show()
    
    print("\nFull analyst notes by team:")
    print("="*70)
    for _, row in df[["team_name","cp8_analyst_note"]].dropna().iterrows():
        print(f"\n[{row['team_name']}]")
        print(row["cp8_analyst_note"])
        print("-"*50)

In [ ]:
#@title 2.4 — Within-class methodological divergence analysis
display(Markdown("""
### 2.4 \u2014 Divergence analysis: does methodology drive outcome?

Key questions for class discussion:
- Did teams choosing **VADER** report more negative crisis-period sentiment than teams using **LM** or **Harvard IV-4**?
- Did teams choosing **FinBERT** report higher accuracy than **DistilBERT** teams?
- Did teams that chose **more topics (higher k)** tend to describe more specific BrewDog themes in their analyst notes, or did smaller k produce more coherent narratives?
- Is there a relationship between **pre-processing strictness** (e.g. lemmatisation + finance-adjusted stop words) and the **sentiment scores** reported?
"""))

if "cp4_dictionary" in df.columns and "cp4_sentiment_2019_2025" in df.columns:
    grp = df.groupby("cp4_dictionary")["cp4_sentiment_2019_2025"].agg(["mean","std","count"])
    grp.columns = ["Mean sentiment (2019\u20132025)","Std dev","Teams"]
    grp["Mean sentiment (2019\u20132025)"] = grp["Mean sentiment (2019\u20132025)"].round(4)
    grp["Std dev"] = grp["Std dev"].round(4)
    print("\nCrisis-period sentiment by dictionary choice:")
    display(grp)